# 03 — DarkBERT on the CoDA Dataset

**Goal:** fine-tune **DarkBERT** — a RoBERTa-base model continue-pretrained on Tor-crawled pages by S2W + KAIST — on the **CoDA** dataset of labeled onion pages. Pure hands-on: load the gated dataset and the gated model and see what happens when you put them together.

## Why this notebook exists

Nb 02 used generic ModernBERT on a drug marketplace dump (Agora). This notebook flips both variables at once — domain-pretrained model, domain-native dataset — so you can feel what happens when the model actually matches the text it's reading.

- **Dataset:** `s2w-ai/CoDA` — 10,000 Tor-crawled onion pages, labeled across 10 categories (Drugs, Hacking, Financial, Arms, Gambling, Crypto, Porn, Violence, Electronic, Others). 51 languages, but 88% English; we use the English slice. Gated access on Hugging Face.
- **Model:** `s2w-ai/DarkBERT` — RoBERTa-base pretrained on a Tor corpus. 512-token context. Gated access on Hugging Face.

## Prerequisites

```bash
pip install -U transformers datasets scikit-learn torch huggingface_hub
huggingface-cli login   # needed for gated access to CoDA and DarkBERT
```

You must have been granted access to both `s2w-ai/CoDA` and `s2w-ai/DarkBERT` on Hugging Face.

## 1 — Build the dataset from CoDA

CoDA ships as a tarball of `.txt` files. The filename carries the label: `<id>-<category>-<lang>-<hash>.txt`. We parse filenames, keep English pages, drop the `Others` bucket (29% of the data — it's a noise class of blank pages and miscellaneous), build a 9-class dataset.

In [1]:
import json, random
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel
from huggingface_hub import hf_hub_download

DATA = Path('data'); DATA.mkdir(exist_ok=True)
PROCESSED = Path('processed'); PROCESSED.mkdir(exist_ok=True)
CODA_DIR = DATA / 'coda' / 'coda_dataset'

# --- Download + extract if not already on disk --------------------------
if not CODA_DIR.exists():
    import tarfile
    print('Downloading CoDA from Hugging Face...')
    tar_path = hf_hub_download(repo_id='s2w-ai/CoDA', filename='coda_dataset.tar.gz',
                                repo_type='dataset', local_dir=str(DATA / 'coda'))
    print('Extracting...')
    with tarfile.open(tar_path) as tf:
        tf.extractall(DATA / 'coda')

files = list(CODA_DIR.glob('*.txt'))
print(f'Total CoDA files: {len(files)}')

# --- Parse filename, filter, load text ---------------------------------
rows = []
for f in files:
    parts = f.stem.split('-')  # id-category-lang-hash
    if len(parts) < 4:
        continue
    _, cat, lang, _ = parts[0], parts[1], parts[2], parts[3]
    if lang != 'en' or cat == 'Others':
        continue
    text = f.read_text(errors='ignore').strip()
    if len(text.split()) < 20:  # drop near-empty pages
        continue
    rows.append({'text': text, 'top': cat})

df = pd.DataFrame(rows)
print(f'After EN + non-Others + text>20 words: {len(df)} rows')
print(f'\nClass counts:')
print(df['top'].value_counts().to_string())

lens = df['text'].str.split().str.len()
print(f'\nText length (words): median={int(lens.median())}, '
      f'p95={int(lens.quantile(0.95))}, max={lens.max()}')
print(f'% of pages over 512 words: {(lens > 512).mean()*100:.0f}%')

# --- Label encoding + stratified 80/10/10 split -------------------------
label_names = sorted(df['top'].unique())
label2id = {n: i for i, n in enumerate(label_names)}
df['label'] = df['top'].map(label2id)

from sklearn.model_selection import train_test_split
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

features = Features({'text': Value('string'), 'label': ClassLabel(names=label_names)})
ds = DatasetDict({
    'train': Dataset.from_pandas(train_df[['text', 'label']], features=features, preserve_index=False),
    'validation': Dataset.from_pandas(val_df[['text', 'label']], features=features, preserve_index=False),
    'test': Dataset.from_pandas(test_df[['text', 'label']], features=features, preserve_index=False),
})
ds.save_to_disk(str(PROCESSED / 'coda_en'))
with open(PROCESSED / 'coda_en' / 'label_names.json', 'w') as f:
    json.dump({'names': label_names, 'label2id': label2id}, f, indent=2)

print(f'\nSaved: {ds}')

Total CoDA files: 10000
After EN + non-Others + text>20 words: 6582 rows

Class counts:
top
Porn          1134
Drugs          948
Financial      924
Gambling       755
Crypto         721
Hacking        616
Arms           595
Violence       474
Electronic     415

Text length (words): median=578, p95=7189, max=18671
% of pages over 512 words: 53%


Saving the dataset (0/1 shards):   0%|          | 0/5265 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/658 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/659 [00:00<?, ? examples/s]


Saved: DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 5265
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 658
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 659
    })
})


**What to notice:** most pages exceed DarkBERT's 512-token window. We'll truncate — the first 512 tokens of an onion page (title + opening paragraphs + nav) usually contain enough signal for category classification. Not ideal, but it's DarkBERT's native limit.

## 2 — Load DarkBERT + tokenize

DarkBERT is a standard RoBERTa-base — the HuggingFace API loads it like any other model. The weights are the only thing that's special, and they're gated.

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_from_disk

assert torch.cuda.is_available(), 'GPU expected.'
print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')

MODEL_CKPT = 's2w-ai/DarkBERT'
MAX_LEN = 512  # DarkBERT's native limit

ds = load_from_disk(str(PROCESSED / 'coda_en'))
with open(PROCESSED / 'coda_en' / 'label_names.json') as f:
    vocab = json.load(f)
label_names = vocab['names']
num_labels = len(label_names)

tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)

tok = ds.map(tokenize, batched=True, remove_columns=['text'])
print(tok)

GPU: NVIDIA GeForce RTX 3070 Laptop GPU (8.2 GB)
DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 5265
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 658
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 659
    })
})


## 3 — Fine-tune DarkBERT

Standard single-label classification recipe: softmax head, cross-entropy, fp16, 3 epochs. Select on macro F1 so the smaller classes (Electronic, Violence) count.

In [3]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=num_labels,
    id2label={i: n for i, n in enumerate(label_names)},
    label2id={n: i for i, n in enumerate(label_names)},
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro', zero_division=0),
        'weighted_f1': f1_score(labels, preds, average='weighted', zero_division=0),
    }

args = TrainingArguments(
    output_dir='models/darkbert_coda',
    num_train_epochs=3,
    per_device_train_batch_size=8,    # 512 tokens × 8 fits in 8 GB with fp16
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok['train'],
    eval_dataset=tok['validation'],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: s2w-ai/DarkBERT
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/home/saqib

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.167480,0.206008,0.955927,0.954909,0.955430
2,0.172982,0.151790,0.972644,0.973341,0.972630
3,0.083925,0.164633,0.968085,0.968932,0.967974


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1977, training_loss=0.2770610920515818, metrics={'train_runtime': 1169.9845, 'train_samples_per_second': 13.5, 'train_steps_per_second': 1.69, 'total_flos': 4156100314629120.0, 'train_loss': 0.2770610920515818, 'epoch': 3.0})

## 4 — Evaluate on test + per-class breakdown

In [4]:
test_out = trainer.predict(tok['test'])
y_true = test_out.label_ids
y_pred = np.argmax(test_out.predictions, axis=-1)

print(f'Test accuracy:    {accuracy_score(y_true, y_pred):.4f}')
print(f'Test macro F1:    {f1_score(y_true, y_pred, average="macro", zero_division=0):.4f}')
print(f'Test weighted F1: {f1_score(y_true, y_pred, average="weighted", zero_division=0):.4f}')

print('\nPer-class breakdown:')
per = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0,
                                       labels=list(range(num_labels)))
for i, name in enumerate(label_names):
    print(f'  {name:12s}  P={per[0][i]:.3f}  R={per[1][i]:.3f}  F1={per[2][i]:.3f}  n={per[3][i]}')

cm = confusion_matrix(y_true, y_pred, labels=list(range(num_labels)))
print(f'\nConfusion matrix (rows=true, cols=predicted):')
print(' ' * 14 + ' '.join(f'{n[:5]:>6s}' for n in label_names))
for i, name in enumerate(label_names):
    print(f'  {name:12s}  ' + ' '.join(f'{cm[i,j]:6d}' for j in range(num_labels)))

Test accuracy:    0.9833
Test macro F1:    0.9820
Test weighted F1: 0.9833

Per-class breakdown:
  Arms          P=0.951  R=0.983  F1=0.967  n=59
  Crypto        P=0.947  R=0.986  F1=0.966  n=72
  Drugs         P=0.989  R=0.979  F1=0.984  n=95
  Electronic    P=1.000  R=1.000  F1=1.000  n=41
  Financial     P=1.000  R=0.957  F1=0.978  n=93
  Gambling      P=1.000  R=1.000  F1=1.000  n=76
  Hacking       P=0.954  R=1.000  F1=0.976  n=62
  Porn          P=1.000  R=1.000  F1=1.000  n=114
  Violence      P=1.000  R=0.936  F1=0.967  n=47

Confusion matrix (rows=true, cols=predicted):
                Arms  Crypt  Drugs  Elect  Finan  Gambl  Hacki   Porn  Viole
  Arms              58      1      0      0      0      0      0      0      0
  Crypto             0     71      0      0      0      0      1      0      0
  Drugs              1      0     93      0      0      0      1      0      0
  Electronic         0      0      0     41      0      0      0      0      0
  Financial          

## 5 — Save + live inference

Save the fine-tuned model + tokenizer + label map for later reuse. Then sanity-check on a few handwritten snippets that look like the opening text of a real onion page.

In [5]:
FINAL = Path('models/darkbert_coda_final')
FINAL.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(FINAL))
tokenizer.save_pretrained(str(FINAL))
with open(FINAL / 'label_names.json', 'w') as f:
    json.dump({'names': label_names, 'label2id': vocab['label2id']}, f, indent=2)
print(f'Saved to {FINAL}')

examples = [
    'Welcome to the marketplace. Buy cocaine, MDMA, LSD with bitcoin. Worldwide shipping, stealth packaging, PGP communication only.',
    'Premium zero-day exploits for sale. RCE in Apache, VMware, Fortinet. Escrow available. Contact via XMPP with PGP key.',
    'Glock 17 and AR-15 lower receivers, untraceable builds, EU shipping from dead drops in Germany and Netherlands.',
    'Verified Bitcoin mixing service, no logs, 0.5% fee, 24-hour cooldown, onion mirror list below.',
    'Counterfeit Euro banknotes, 50 and 100 denominations, passes UV and pen tests, sample order available.',
    'Provably fair dice and blackjack, instant bitcoin deposits, no KYC, referral program 20%.',
]

model.eval()
device = next(model.parameters()).device
print('\nLive inference:')
for ex in examples:
    enc = tokenizer(ex, truncation=True, max_length=MAX_LEN, return_tensors='pt').to(device)
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits[0], dim=-1).cpu().numpy()
    top3 = probs.argsort()[-3:][::-1]
    print(f'\n  {ex[:70]}...')
    for i in top3:
        print(f'    {label_names[i]:12s}  {probs[i]:.3f}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to models/darkbert_coda_final

Live inference:

  Welcome to the marketplace. Buy cocaine, MDMA, LSD with bitcoin. World...
    Drugs         0.998
    Crypto        0.000
    Arms          0.000

  Premium zero-day exploits for sale. RCE in Apache, VMware, Fortinet. E...
    Hacking       0.995
    Crypto        0.002
    Financial     0.001

  Glock 17 and AR-15 lower receivers, untraceable builds, EU shipping fr...
    Arms          0.996
    Electronic    0.001
    Violence      0.001

  Verified Bitcoin mixing service, no logs, 0.5% fee, 24-hour cooldown, ...
    Crypto        0.997
    Financial     0.001
    Hacking       0.000

  Counterfeit Euro banknotes, 50 and 100 denominations, passes UV and pe...
    Financial     0.996
    Crypto        0.001
    Electronic    0.000

  Provably fair dice and blackjack, instant bitcoin deposits, no KYC, re...
    Gambling      0.998
    Crypto        0.000
    Arms          0.000


## What you've done

- Pulled the gated **CoDA** dataset from Hugging Face, parsed its filename-encoded labels, and built a clean 9-class English-only classification set.
- Fine-tuned **DarkBERT** — a RoBERTa-base model pretrained on Tor pages — on that data.
- Measured how well a dark-web-native model categorizes dark-web-native text.
